# Pokémon Battle Outcome Predictor

## Problem Statement
Professor Oak's lab has been collecting data from thousands of Pokémon battles.  
Each record contains the **base stats** (HP, Attack, Defense, Sp. Atk, Sp. Def, Speed) and **type** of both Pokémon, along with which one won.

Your mission: build two classifiers to predict the winner.

1. **DecisionTreeClassifier** — a simple tree-based baseline  
2. **XGBClassifier** — a gradient-boosted ensemble

Both models will be evaluated on a **held-out private test set** using **accuracy**.  
A **public inference set** is provided so you can verify your models locally before submitting.

## Evaluation
| Test Case | Model | Threshold |
|-----------|-------|-----------|
| TC1 | DecisionTree | accuracy ≥ 0.78 |
| TC2 | DecisionTree | accuracy ≥ 0.85 |
| TC3 | XGBClassifier | accuracy ≥ 0.85 |
| TC4 | XGBClassifier | accuracy ≥ 0.90 |
| TC5 | XGBClassifier | accuracy ≥ 0.92 |

In [ ]:
# ================= Constants =================
BOILERPLATE_OBJECTS    = '../../objects/boilerplate'
SOLUTION_OBJECTS       = '../../objects/solution'

TRAIN_DATA_PATH            = BOILERPLATE_OBJECTS + '/datasets/training/training_data.csv'
PUBLIC_INFERENCE_DATA_PATH = BOILERPLATE_OBJECTS + '/datasets/inference/inference_data.csv'
TREE_MODEL_PATH            = SOLUTION_OBJECTS + '/learned_weights/decision_tree.pkl'
XGB_MODEL_PATH             = SOLUTION_OBJECTS + '/learned_weights/xgboost_model.pkl'

# Pokémon type indices
# 0=Normal, 1=Fire, 2=Water, 3=Grass, 4=Electric, 5=Ice, 6=Fighting, 7=Poison,
# 8=Ground, 9=Flying, 10=Psychic, 11=Bug, 12=Rock, 13=Ghost, 14=Dragon,
# 15=Dark, 16=Steel, 17=Fairy

# Type effectiveness chart: (attacker_type, defender_type) -> multiplier
# 2.0 = super effective, 0.5 = not very effective, 0.0 = immune, 1.0 = neutral (default)
# Hint: This chart might be useful for creating features that capture type matchup dynamics.
TYPE_CHART = {
    (1,3):2.0,(1,5):2.0,(1,11):2.0,(1,16):2.0,(1,2):0.5,(1,12):0.5,(1,1):0.5,(1,14):0.5,
    (2,1):2.0,(2,8):2.0,(2,12):2.0,(2,2):0.5,(2,3):0.5,(2,14):0.5,
    (3,2):2.0,(3,8):2.0,(3,12):2.0,(3,1):0.5,(3,3):0.5,(3,7):0.5,(3,9):0.5,(3,11):0.5,(3,14):0.5,(3,16):0.5,
    (4,2):2.0,(4,9):2.0,(4,3):0.5,(4,4):0.5,(4,14):0.5,(4,8):0.0,
    (5,3):2.0,(5,8):2.0,(5,9):2.0,(5,14):2.0,(5,1):0.5,(5,2):0.5,(5,5):0.5,(5,16):0.5,
    (6,0):2.0,(6,5):2.0,(6,12):2.0,(6,15):2.0,(6,16):2.0,(6,7):0.5,(6,9):0.5,(6,10):0.5,(6,11):0.5,(6,17):0.5,(6,13):0.0,
    (10,6):2.0,(10,7):2.0,(10,10):0.5,(10,16):0.5,(10,15):0.0,
    (14,14):2.0,(14,16):0.5,(14,17):0.0,
    (13,10):2.0,(13,13):2.0,(13,15):0.5,(13,0):0.0,
    (15,10):2.0,(15,13):2.0,(15,6):0.5,(15,15):0.5,(15,17):0.5,
    (16,5):2.0,(16,12):2.0,(16,17):2.0,(16,1):0.5,(16,2):0.5,(16,4):0.5,(16,16):0.5,
    (17,6):2.0,(17,14):2.0,(17,15):2.0,(17,1):0.5,(17,7):0.5,(17,16):0.5,
    (9,3):2.0,(9,6):2.0,(9,11):2.0,(9,4):0.5,(9,12):0.5,(9,16):0.5,
    (12,1):2.0,(12,5):2.0,(12,9):2.0,(12,11):2.0,(12,6):0.5,(12,8):0.5,(12,16):0.5,
    (8,1):2.0,(8,4):2.0,(8,7):2.0,(8,12):2.0,(8,16):2.0,(8,3):0.5,(8,11):0.5,(8,9):0.0,
    (11,3):2.0,(11,10):2.0,(11,15):2.0,(11,1):0.5,(11,6):0.5,(11,7):0.5,(11,9):0.5,(11,13):0.5,(11,16):0.5,(11,17):0.5,
    (7,3):2.0,(7,17):2.0,(7,7):0.5,(7,8):0.5,(7,12):0.5,(7,13):0.5,(7,16):0.0,
    (0,12):0.5,(0,16):0.5,(0,13):0.0,
}

In [ ]:
# ================= Imports =================
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

In [ ]:
# ================= Device Detection =================
import subprocess

def get_xgb_device():
    """Detect best available device for XGBoost: CUDA GPU > CPU."""
    try:
        result = subprocess.run(['nvidia-smi'], capture_output=True, timeout=5)
        if result.returncode == 0:
            return 'cuda'
    except (FileNotFoundError, subprocess.TimeoutExpired):
        pass
    return 'cpu'

DEVICE = get_xgb_device()
print(f"XGBoost device: {DEVICE}")

In [ ]:
# ================= Load Data =================
df_train = pd.read_csv(TRAIN_DATA_PATH)
X_train  = df_train.drop('winner', axis=1)
y_train  = df_train['winner'].values

print(f"Training samples : {X_train.shape[0]}")
print(f"Features         : {X_train.shape[1]}")
print(f"Feature names    : {list(X_train.columns)}")
print(f"Winner dist      : {dict(zip(*np.unique(y_train, return_counts=True)))}")

# Quick look at stat distributions
fig, axes = plt.subplots(2, 7, figsize=(18, 5))
for i, col in enumerate(X_train.columns):
    ax = axes[i // 7][i % 7]
    ax.hist(X_train[col], bins=30, alpha=0.7)
    ax.set_title(col, fontsize=8)
    ax.tick_params(labelsize=6)
plt.suptitle('Feature Distributions')
plt.tight_layout()
plt.show()

## Part 1 — DecisionTreeClassifier (Baseline)

Start with a `DecisionTreeClassifier`. This is a simple, interpretable model that splits the feature space using axis-aligned cuts.  
Consider whether any feature engineering could help the tree find better splits.  
Return the trained model and the list of feature names used (so inference can reconstruct the same features).

In [ ]:
# ================= Train DecisionTree (YOUR CODE HERE) =================
def engineer_features(df):
    """
    Create derived features from raw battle stats.
    Returns a new DataFrame with original + engineered features.
    """
    df = df.copy()
    # Raw stats tell you each Pokémon's strength in isolation.
    # But battles are about match-ups — who has the *edge*?
    # Think: what matters more, a Pokémon's raw attack, or how much
    # it exceeds the opponent's defense?

    # Also consider: the p1_type and p2_type columns are just integers.
    # A Fire-type attacking a Grass-type has a very different outcome than
    # attacking a Water-type. Can you turn that domain knowledge into a feature?
    # (The TYPE_CHART constant above might come in handy...)

    # YOUR CODE HERE
    return df


def train_decision_tree(X_train, y_train):
    """
    Train a DecisionTreeClassifier.

    Args:
        X_train : pd.DataFrame with battle features
        y_train : np.ndarray, shape (n_samples,)

    Returns:
        feature_names : list of feature column names used
        model         : trained DecisionTreeClassifier
    """
    X_eng = engineer_features(X_train)
    feature_names = list(X_eng.columns)

    # A deep tree memorizes the training data; a shallow tree underfits.
    # Try tuning max_depth and min_samples_leaf to find the sweet spot.
    model = DecisionTreeClassifier(random_state=42)

    # YOUR CODE HERE
    return feature_names, model


tree_features, tree_model = train_decision_tree(X_train, y_train)
print("DecisionTree training complete.")

## Part 2 — XGBClassifier

Now train an `XGBClassifier`. XGBoost builds an ensemble of decision trees sequentially, where each new tree corrects the errors of the previous ones.  
Think about how feature engineering and hyperparameter tuning can improve accuracy over the baseline.

In [ ]:
# ================= Train XGBClassifier (YOUR CODE HERE) =================
def train_xgboost(X_train, y_train):
    """
    Train an XGBClassifier.

    Args:
        X_train : pd.DataFrame with battle features
        y_train : np.ndarray, shape (n_samples,)

    Returns:
        feature_names : list of feature column names used
        model         : trained XGBClassifier
    """
    X_eng = engineer_features(X_train)
    feature_names = list(X_eng.columns)

    # Key hyperparameters to experiment with:
    #   n_estimators  — more trees can help, but too many risks overfitting
    #   max_depth     — controls individual tree complexity
    #   learning_rate — smaller values need more trees but generalize better
    #   subsample / colsample_bytree — adds randomness to reduce overfitting
    model = XGBClassifier(device=DEVICE, random_state=42, eval_metric='logloss')

    # YOUR CODE HERE
    return feature_names, model


xgb_features, xgb_model = train_xgboost(X_train, y_train)
print("XGBClassifier training complete.")

In [ ]:
# ================= Compare on Public Inference Set =================
df_infer = pd.read_csv(PUBLIC_INFERENCE_DATA_PATH)
X_infer  = df_infer.drop('winner', axis=1)
y_infer  = df_infer['winner'].values

def evaluate(model, feature_names, X, y, label):
    X_eng = engineer_features(X)
    X_proc = X_eng[feature_names].values
    acc = np.mean(model.predict(X_proc) == y)
    print(f"{label:<25} accuracy: {acc:.4f}")
    return acc

acc_tree = evaluate(tree_model, tree_features, X_infer, y_infer, 'DecisionTree')
acc_xgb  = evaluate(xgb_model, xgb_features, X_infer, y_infer, 'XGBClassifier')
print(f"\nAccuracy delta (XGB - Tree): {acc_xgb - acc_tree:+.4f}")

In [ ]:
# ================= Save Artifacts =================
os.makedirs(os.path.dirname(TREE_MODEL_PATH), exist_ok=True)

joblib.dump({'model': tree_model, 'feature_names': tree_features}, TREE_MODEL_PATH)
joblib.dump({'model': xgb_model, 'feature_names': xgb_features}, XGB_MODEL_PATH)

print(f"DecisionTree saved to  {TREE_MODEL_PATH}")
print(f"XGBClassifier saved to {XGB_MODEL_PATH}")